In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# example_idx in token_scores.npz is a row index into this exact tokenized
# dataset (it's what --dataset pointed at when the scores were computed), so
# this is the dataset to use for recovering prompts, not the raw career.jsonl.
ds = Dataset.load_from_disk("../../../datasets_tokenized/tokenized_career_olmo")
tokenizer = AutoTokenizer.from_pretrained("allenai/Olmo-3-7B-Instruct-SFT")
print("columns:", ds.column_names, "| rows:", len(ds))

In [ ]:
def show_prompt_completion(ds, example_idx: int, tokenizer) -> None:
    """Print one document's prompt and completion as plain text.

    Each row's `input_ids` is the full prompt+completion sequence; `labels`
    is the same length with -100 at every prompt position (the standard
    completion-only-loss mask) and the real token id at every completion
    position. Splitting on that boundary recovers each half, and decoding
    each whole slice at once (not token-by-token) gives readable text.
    """
    example = ds[example_idx]
    input_ids = example["input_ids"]
    labels = example["labels"]
    prompt_ids = [i for i, l in zip(input_ids, labels) if l == -100]
    completion_ids = [i for i, l in zip(input_ids, labels) if l != -100]
    print(f"=== example_idx {example_idx} ===")
    print("--- PROMPT ---")
    print(tokenizer.decode(prompt_ids))
    print()
    print("--- COMPLETION ---")
    print(tokenizer.decode(completion_ids))


show_prompt_completion(ds, 0, tokenizer)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors
from IPython.display import HTML, display

# Same npz used in tokenIFscores.ipynb's concentration/top_tokens analysis.
d = np.load("../../../scped_IFS/token_scores_olmo_career.npz")
scores_df = pd.DataFrame({k: d[k] for k in d.files})
print("scored tokens:", len(scores_df), "| unique documents:", scores_df["example_idx"].nunique())

In [ ]:
import html


def highlighted_completion_html(scores_df: pd.DataFrame, example_idx: int, tokenizer) -> str:
    """The completion's tokens as HTML spans colored by score: blue = negative
    (reduces misalignment), red = positive (drives misalignment)."""
    rows = scores_df[scores_df["example_idx"] == example_idx].sort_values("position")
    scores = rows["score"].to_numpy()
    vmax = np.abs(scores).max() or 1.0
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    cmap = matplotlib.colormaps["RdBu_r"]
    spans = []
    for token_id, score in zip(rows["token_id"], scores):
        # Escaped: decoded text can contain chat-template special tokens like
        # <|im_start|> or <functions></functions>, which a browser would
        # otherwise try to parse as real HTML tags and silently hide.
        text = html.escape(tokenizer.decode([token_id]))
        r, g, b, _ = cmap(norm(score))
        color = f"rgba({int(r*255)},{int(g*255)},{int(b*255)},0.65)"
        spans.append(f'<span style="background-color:{color}" title="{score:.1f}">{text}</span>')
    return "".join(spans)


def document_html(ds, scores_df: pd.DataFrame, example_idx: int, tokenizer) -> str:
    """HTML for one document: plain-text prompt, then its score-highlighted
    completion. Shared by the inline notebook display and the file export,
    so both always show exactly the same thing."""
    example = ds[example_idx]
    prompt_ids = [i for i, l in zip(example["input_ids"], example["labels"]) if l == -100]
    prompt_html = html.escape(tokenizer.decode(prompt_ids)).replace("\n", "<br>")
    completion_html = highlighted_completion_html(scores_df, example_idx, tokenizer)
    return (
        f"<h4>example_idx {example_idx}</h4>"
        f"<div style='color:#555'>{prompt_html}</div>"
        f"<div style='margin-top:6px'>{completion_html}</div>"
        f"<hr>"
    )


def show_document(ds, scores_df: pd.DataFrame, example_idx: int, tokenizer) -> None:
    """Display one document inline in the notebook."""
    display(HTML(document_html(ds, scores_df, example_idx, tokenizer)))


def top_example_indices(scores_df: pd.DataFrame, n: int = 15) -> list[int]:
    """The documents containing the n highest-scoring individual tokens, in
    score order, deduplicated (top tokens are almost always from distinct
    documents — see the check above: 15/15 unique at n=15)."""
    seen = []
    for idx in scores_df.sort_values("score", ascending=False)["example_idx"]:
        if idx not in seen:
            seen.append(idx)
        if len(seen) == n:
            break
    return seen


for example_idx in top_example_indices(scores_df, n=15):
    show_document(ds, scores_df, example_idx, tokenizer)

In [ ]:
from pathlib import Path


def save_documents_html(ds, scores_df: pd.DataFrame, example_indices: list[int], tokenizer, path) -> None:
    """Write a standalone HTML file with the given documents, preserving the
    same red/blue score-highlighting as the inline notebook display — open
    it directly in a browser, or share it as-is."""
    body = "\n".join(document_html(ds, scores_df, idx, tokenizer) for idx in example_indices)
    page = (
        "<html><head><meta charset='utf-8'>"
        "<title>Token influence — highlighted examples</title>"
        "<style>body{font-family:sans-serif;max-width:800px;margin:40px auto;line-height:1.6}"
        "h4{margin-bottom:4px}</style></head><body>" + body + "</body></html>"
    )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(page)
    print("wrote", path)


save_documents_html(
    ds, scores_df, top_example_indices(scores_df, n=15), tokenizer,
    "/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/outputs/docs/top_influence_tokens_career.html",
)